In [7]:
import anndata as ad
import pandas as pd
import numpy as np
import tables
import h5py

In [2]:
path = "/cluster/work/boeva/eheiss/datasets/ARCHS4/human_gene_v2.latest.h5"

with h5py.File(path, "r") as f:
    print(list(f.keys()))

['data', 'meta']


In [3]:
def print_tree(name, obj):
    print(name, type(obj))

with h5py.File(path, "r") as f:
    f.visititems(print_tree)

data <class 'h5py._hl.group.Group'>
data/expression <class 'h5py._hl.dataset.Dataset'>
meta <class 'h5py._hl.group.Group'>
meta/genes <class 'h5py._hl.group.Group'>
meta/genes/biotype <class 'h5py._hl.dataset.Dataset'>
meta/genes/ensembl_gene <class 'h5py._hl.dataset.Dataset'>
meta/genes/symbol <class 'h5py._hl.dataset.Dataset'>
meta/info <class 'h5py._hl.group.Group'>
meta/info/author <class 'h5py._hl.dataset.Dataset'>
meta/info/contact <class 'h5py._hl.dataset.Dataset'>
meta/info/creation-date <class 'h5py._hl.dataset.Dataset'>
meta/info/laboratory <class 'h5py._hl.dataset.Dataset'>
meta/info/version <class 'h5py._hl.dataset.Dataset'>
meta/samples <class 'h5py._hl.group.Group'>
meta/samples/alignedreads <class 'h5py._hl.dataset.Dataset'>
meta/samples/channel_count <class 'h5py._hl.dataset.Dataset'>
meta/samples/characteristics_ch1 <class 'h5py._hl.dataset.Dataset'>
meta/samples/contact_address <class 'h5py._hl.dataset.Dataset'>
meta/samples/contact_city <class 'h5py._hl.dataset.Datas

In [4]:
with h5py.File(path, "r") as f:
    sc_prob = f["meta/samples/singlecellprobability"][:10]

print(sc_prob)

[0.         0.         0.         0.00489635 0.         0.00056879
 0.23247727 0.26097082 0.3147339  0.18901751]


In [5]:
with h5py.File(path, "r") as f:
    print("expression:", f["data/expression"].shape, f["data/expression"].dtype)
    print("genes:", f["meta/genes/symbol"].shape, f["meta/genes/symbol"].dtype)
    print("samples:", f["meta/samples/sample"].shape, f["meta/samples/sample"].dtype)

expression: (67186, 1075320) uint32
genes: (67186,) object
samples: (1075320,) object


In [12]:
with h5py.File(path, "r") as f:
    ensembl = f["meta/genes/ensembl_gene"][:]

# decode bytes → string
ensembl = [e.decode() if isinstance(e, bytes) else e for e in ensembl]

In [13]:
tcga = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad")
gtex = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/GTEx/gtex.h5ad")
gdsc = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/GDSC/gdsc.h5ad")
depmap = ad.read_h5ad("/cluster/work/boeva/eheiss/datasets/DepMap/depmap.h5ad")
adata = ad.concat([tcga, gtex, gdsc, depmap], join = "inner")

In [14]:
genes = set(adata.var_names)

In [16]:
archs4_genes = set(ensembl)

In [19]:
len(archs4_genes & genes)

18397

In [20]:
len(genes)

18397

In [22]:
with h5py.File(path, "r") as f:
    X = f["data/expression"]
    
    # take 5 samples
    chunk = X[:, :5].T   # (5 samples, 67186 genes)

print(chunk)

[[ 105    0 3028 ...    0    0    0]
 [  77    0 2173 ...    0    0    0]
 [  90    0 2915 ...    0    0    0]
 [ 203    0 3525 ...    0    0    0]
 [ 174    1 3082 ...    0    0    2]]
